# Workflow tiếng Việt cho project Checkout the Canteen

Notebook này ghi rõ file code nào dùng để làm gì. Chạy từng cell theo nhu cầu, không nhất thiết chạy hết từ đầu tới cuối.

## 1. Kiểm tra môi trường

- `scripts/07_smoke_test.py`: kiểm tra import thư viện, GPU PyTorch, OpenCV đọc/ghi ảnh.
- Dùng khi mới clone repo hoặc đổi sang Kaggle/Colab/local khác.

In [ ]:
!python scripts/07_smoke_test.py

## 2. Audit data hiện tại

- `scripts/20_audit_dataset_conflicts.py`: tìm ảnh lỗi, ảnh trùng SHA, ảnh gần giống pHash nhưng nằm ở class khác.
- Mặc định chỉ xuất report, không move ảnh.
- Nếu muốn đưa conflict vào quarantine thì thêm `--quarantine`; nên chạy dry-run trước.

In [ ]:
!python scripts/20_audit_dataset_conflicts.py --root data/classification --phash-threshold 4
!python scripts/20_audit_dataset_conflicts.py --root data/classification --phash-threshold 4 --quarantine --dry-run

## 3. Mở Data IDE để lọc trực tiếp

- `scripts/21_data_ide.py`: giao diện quản lý data như một IDE nhỏ.
- Chọn `data/classification`, `external_review`, `external_reviewed`, hoặc `quarantine`.
- Có thể move class, quarantine, undo, chạy model để xem top1/top2/confidence/margin.
- Mở trình duyệt tại `http://127.0.0.1:7862`.

In [ ]:
!python scripts/21_data_ide.py --host 127.0.0.1 --port 7862

## 4. Tìm dataset public trước khi crawl ảnh lẻ

- `scripts/22_search_public_datasets.py`: tìm link dataset Kaggle/GitHub/Roboflow/HuggingFace liên quan.
- Output là manifest CSV để mình đọc và quyết định dataset nào đáng tải.

In [ ]:
!python scripts/22_search_public_datasets.py --provider mixed --max-results 10

## 5. Crawl ảnh candidate khi thiếu data

- `scripts/09_collect_web_images.py`: crawl ảnh candidate từ query CSV.
- Nên dùng `--provider mixed` để lấy DuckDuckGo + Bing, rồi dedupe SHA/pHash.
- Không crawl thẳng vào train; luôn để ở batch raw rồi import/preprocess/review.

In [ ]:
!python scripts/09_collect_web_images.py --queries configs/green_vegetable_extra_queries.csv --provider mixed --out data/downloads/scrape_batches/new_batch/raw --manifest data/downloads/scrape_batches/new_batch/scraped_manifest.csv --per-query 80 --max-downloads-per-class 300 --dedupe-against data/classification --dedupe-against data/downloads --phash-threshold 6

## 6. Import/preprocess batch đã lọc

- `scripts/18_import_scrape_batch_to_review.py`: chuẩn hóa ảnh về JPEG 512x512, reject ảnh lỗi/mờ/quá trùng.
- Dùng `--target review` nếu còn muốn lọc tay.
- Dùng `--target reviewed` nếu bạn đã tự xoá rác trong raw batch rồi.

In [ ]:
!python scripts/18_import_scrape_batch_to_review.py --source data/downloads/scrape_batches/new_batch/raw --class-name rau_xao --pool rau_xao_new_batch --target review --dedupe-against data/classification

## 7. Build lại data/classification

- `scripts/17_build_weighted_classification_dataset.py`: merge old + reviewed thành `data/classification`.
- Hiện tại `old` và `reviewed` đều weight 1, priority ngang nhau.
- Cross-class conflict sẽ bị skip và ghi report.

In [ ]:
!python scripts/17_build_weighted_classification_dataset.py --old-weight 1 --reviewed-weight 1 --cross-class-hamming 4 --clear

## 8. Train model

- `scripts/05_train_classifier.py`: train classifier 11 món.
- Dùng EfficientNet-B0, epoch ngắn và label smoothing để giảm overfit.
- Script lưu best checkpoint theo validation, không dùng epoch cuối nếu epoch cuối tệ hơn.

In [ ]:
!python scripts/05_train_classifier.py --arch efficientnet_b0 --epochs 6 --batch-size 8 --lr 0.0001 --label-smoothing 0.05

## 9. Đọc report train

- `outputs/reports/classification_report.txt`: precision/recall/F1 từng món.
- `outputs/reports/training_history.json`: loss/accuracy từng epoch.
- `outputs/reports/confusion_matrix.png`: class nào hay nhầm với class nào.

In [ ]:
!type outputs/reports/classification_report.txt

## 10. Demo checkout

- `scripts/19_demo_checkout_app.py`: giao diện demo hóa đơn, chọn ảnh, chỉnh crop, ignore vùng không tính tiền.
- Mở `http://127.0.0.1:7861`.

In [ ]:
!python scripts/19_demo_checkout_app.py --host 127.0.0.1 --port 7861